# GrIMP Joblib Time-Series Inversion

This notebook demonstrates a complete time-series inversion workflow for the Petermann GrIMP subset NetCDF file. It uses `ice.Stack` to access `VelocityMap(time, band, y, x)`, tunes regularization at selected grid points, then runs the joblib-backed grid inversion for the first three velocity bands: `vx`, `vy`, and `vv`.

The final output is a single NetCDF file with variables `full`, `secular`, `seasonal`, `transient`, and `sigma`, each shaped `(time, band, y, x)` and readable with `ice.Stack(output_nc, indexers={'band': 0})`.

In [ ]:
from pathlib import Path
import os
import shutil
import sys

os.environ.setdefault("MPLCONFIGDIR", str(Path(".matplotlib-cache").resolve()))

for candidate in (Path.cwd(), Path.cwd().parent, Path("/Users/briel/src/iceutils")):
    if (candidate / "iceutils" / "__init__.py").exists():
        sys.path.insert(0, str(candidate))
        break

import h5netcdf
import h5py
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

import iceutils as ice
print("Using iceutils from", ice.__file__)
from iceutils.raster import get_chunks
from iceutils.stack import TIME_CALENDAR, TIME_UNITS


## Configuration

Edit this cell for the path, bands, model tuning, selected points, and joblib processor count. The full-grid inversion is guarded by `RUN_FULL_GRID`; leave it as `False` while exploring regularization.

In [ ]:
INPUT_NC = Path("/Users/briel/data/glaciers/greenland/petermann/data/GrIMPSubset.NSIDC-0731.nc")
OUTPUT_NC = INPUT_NC.with_name("GrIMPSubset.NSIDC-0731.joblib_inversion.nc")
TEMP_DIR = INPUT_NC.with_name("GrIMPSubset.NSIDC-0731.joblib_inversion_tmp")

DATA_KEY = "VelocityMap"
BANDS_TO_INVERT = ["vx", "vy", "vv"]
TUNING_BAND = "vv"
ERROR_BAND_FOR = {"vx": "ex", "vy": "ey", "vv": "ev"}
MODEL_KWARGS = {"poly": 1, "bsplines": [32, 16, 8, 4], "periods": []}

SOLVER_TYPE = "ridge"
REG_PARAM_GRID = np.array([0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0], dtype=float)
REG_PARAM = 1.0
PRIOR_COV = False
NT_OUT = 200
N_MIN = 20
N_ITER = 1
N_STD = 3.0

N_PROC = 8
RUN_FULL_GRID = False
FORCE_REBUILD_INPUTS = False
OVERWRITE_OUTPUT = False
CLEAN_TEMP = False

UNCERTAINTY_MIN = 1.0
UNCERTAINTY_MAX = np.inf

# Set to None to auto-suggest three finite points, or replace with row/col pairs.
SELECTED_POINTS = None
# SELECTED_POINTS = [
#     {"name": "point_a", "row": 250, "col": 450},
#     {"name": "point_b", "row": 300, "col": 600},
#     {"name": "point_c", "row": 350, "col": 750},
# ]


## Inspect the GrIMP NetCDF

In [ ]:
assert INPUT_NC.exists(), INPUT_NC

with xr.open_dataset(INPUT_NC, engine="h5netcdf") as ds:
    band_labels = [str(value) for value in ds["band"].values]
    band_index = {label: index for index, label in enumerate(band_labels)}
    print(ds)
    print("Band labels:", band_labels)

required_bands = set(BANDS_TO_INVERT) | set(ERROR_BAND_FOR.values())
missing_bands = sorted(required_bands - set(band_index))
assert not missing_bands, f"Missing expected bands: {missing_bands}"

with ice.Stack(str(INPUT_NC), indexers={"band": band_index[BANDS_TO_INVERT[0]]}) as stack:
    velocity = stack[DATA_KEY]
    print("Selected stack shape:", velocity.shape)
    print("Selected stack dims:", velocity.dims)
    print("Decimal-year span:", float(stack.tdec[0]), "to", float(stack.tdec[-1]))
    print("Raster shape:", stack.hdr.shape)


## Temporal Model

The point-tuning and full-grid inversion cells build the temporal model with the `ice.tseries.build_temporal_model(...)` convenience function. With `PRIOR_COV = False`, `REG_PARAM` is the scalar penalty passed to the solver. If you pass an explicit covariance matrix through `PRIOR_COV`, it is inverted and used as the regularization matrix.

In [ ]:
with ice.Stack(str(INPUT_NC), indexers={"band": band_index[TUNING_BAND]}) as stack:
    model = ice.tseries.build_temporal_model(stack.tdec, **MODEL_KWARGS)

print("Design matrix shape:", model.G.shape)
print("Secular columns:", len(model.isecular))
print("Seasonal columns:", len(model.iseasonal))
print("Transient columns:", len(model.itransient))
print("Regularized columns:", len(model.reg_indices))


## Helper Functions

In [ ]:
COMPONENT_KEYS = ("full", "secular", "seasonal", "transient", "sigma")


def stack_for_band(band_name):
    return ice.Stack(str(INPUT_NC), indexers={"band": band_index[band_name]})


def uncertainty_to_weights(error):
    error = np.asarray(error, dtype=np.float32)
    invalid = (~np.isfinite(error)) | (error <= 0.0)
    clipped = np.clip(error, UNCERTAINTY_MIN, UNCERTAINTY_MAX)
    weights = (1.0 / clipped).astype(np.float32)
    weights[invalid] = np.nan
    return weights


def point_series(band_name, row, col):
    error_band = ERROR_BAND_FOR[band_name]
    with stack_for_band(band_name) as data_stack, stack_for_band(error_band) as error_stack:
        data = data_stack.timeseries(coord=(row, col), key=DATA_KEY).astype(np.float64)
        error = error_stack.timeseries(coord=(row, col), key=DATA_KEY).astype(np.float64)
        weights = uncertainty_to_weights(error)
        invalid = (~np.isfinite(data)) | (~np.isfinite(weights))
        data[invalid] = np.nan
        weights[invalid] = np.nan
        return data, weights, data_stack.tdec.copy()


def fit_point_for_penalty(band_name, row, col, penalty):
    data, weights, tdec = point_series(band_name, row, col)
    data_model = ice.tseries.build_temporal_model(tdec, **MODEL_KWARGS)
    tfit = np.linspace(tdec[0], tdec[-1], NT_OUT)
    output_model = ice.tseries.build_temporal_model(tfit, **MODEL_KWARGS)
    solver = ice.tseries.select_solver(
        SOLVER_TYPE,
        reg_indices=output_model.reg_indices,
        penalty=penalty,
        n_min=N_MIN,
    )
    d_work = data.copy()
    w_work = weights.copy()
    status, coeff, coeff_cov = ice.tseries.iterate_lsqr(
        solver, data_model, data_model.G, d_work, w_work,
        n_iter=N_ITER, n_std=N_STD,
    )
    if status == ice.FAIL:
        return None

    pred = output_model.predict(coeff, sigma=False)
    data_pred = data_model.predict(coeff, sigma=False)["full"]
    valid = np.isfinite(data) & np.isfinite(data_pred)
    rmse = np.sqrt(np.nanmean((data[valid] - data_pred[valid]) ** 2)) if valid.any() else np.nan
    return {
        "penalty": penalty,
        "tdec": tdec,
        "tfit": tfit,
        "data": data,
        "weights": weights,
        "prediction": pred,
        "rmse": rmse,
    }


def suggest_points(band_name="vv", n_points=3):
    with stack_for_band(band_name) as stack:
        mean_map = stack.mean(key=DATA_KEY)
    valid = np.argwhere(np.isfinite(mean_map))
    if valid.size == 0:
        return []
    picks = []
    for index, fraction in enumerate(np.linspace(0.2, 0.8, n_points)):
        row, col = valid[int(fraction * (len(valid) - 1))]
        picks.append({"name": f"point_{index + 1}", "row": int(row), "col": int(col)})
    return picks


## Select Points for Regularization Tuning

The map below uses the mean tuning band, `TUNING_BAND`. Edit `SELECTED_POINTS` in the configuration cell if you want specific row/column locations.

In [ ]:
points = suggest_points(TUNING_BAND) if SELECTED_POINTS is None else SELECTED_POINTS
print(points)

with stack_for_band(TUNING_BAND) as stack:
    mean_tuning_band = stack.mean(key=DATA_KEY)

fig, ax = plt.subplots(figsize=(10, 6))
image = ax.imshow(mean_tuning_band, cmap="viridis")
plt.colorbar(image, ax=ax, label=f"Mean {TUNING_BAND}")
for point in points:
    ax.plot(point["col"], point["row"], "ro")
    ax.text(point["col"] + 5, point["row"] + 5, point["name"], color="white")
ax.set_title(f"Mean {TUNING_BAND} with selected tuning points")
ax.set_xlabel("Column")
ax.set_ylabel("Row")
plt.show()


## Sweep Regularization at Selected Points

The regularization sweep is performed only for `TUNING_BAND`; the chosen `REG_PARAM` is then used for all bands in the full-grid inversion.

In [ ]:
tuning_results = {}

for point in points:
    key = (TUNING_BAND, point["name"])
    tuning_results[key] = []
    for penalty in REG_PARAM_GRID:
        result = fit_point_for_penalty(TUNING_BAND, point["row"], point["col"], float(penalty))
        if result is not None:
            tuning_results[key].append(result)

for key, results in tuning_results.items():
    band_name, point_name = key
    print(f"{band_name} {point_name}")
    for result in results:
        print(f"  penalty={result['penalty']:g}, rmse={result['rmse']:.3f}")


In [ ]:
for key, results in tuning_results.items():
    if not results:
        continue
    band_name, point_name = key
    ncols = min(3, len(results))
    nrows = int(np.ceil(len(results) / ncols))
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 3.5 * nrows), squeeze=False)
    fig.suptitle(f"{band_name} {point_name}")
    for ax, result in zip(axes.ravel(), results):
        pred = result["prediction"]
        ax.plot(result["tdec"], result["data"], "k.", label="data")
        ax.plot(result["tfit"], pred["full"], label="full")
        ax.plot(result["tfit"], pred["secular"], label="secular")
        ax.plot(result["tfit"], pred["seasonal"], label="seasonal")
        ax.plot(result["tfit"], pred["transient"], label="transient")
        ax.set_title(f"penalty={result['penalty']:g}, rmse={result['rmse']:.2f}")
        ax.set_xlabel("Year")
    for ax in axes.ravel()[len(results):]:
        ax.axis("off")
    axes[0, 0].legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.show()


After inspecting the point fits, set `REG_PARAM` in the configuration cell. Then set `RUN_FULL_GRID = True` before running the next cells.

## Build Weighted Per-Band Input Stacks

`ice.tseries.inversion` expects the observation dataset and optional `weights` dataset in the input stack. This helper creates temporary NetCDF stacks for each velocity band with `data` and `weights` variables.

In [ ]:
def _weighted_input_stack_status(path, reference_tdec):
    if not path.exists():
        return False, "missing"
    try:
        with ice.Stack(str(path)) as candidate:
            missing = [name for name in ("data", "weights") if name not in candidate.ds]
            if missing:
                return False, "missing " + ", ".join(missing)
            tdec = np.asarray(candidate.tdec, dtype=float)
            reference_tdec = np.asarray(reference_tdec, dtype=float)
            if tdec.shape != reference_tdec.shape:
                return False, f"time shape {tdec.shape} != {reference_tdec.shape}"
            if not np.all(np.isfinite(tdec)):
                return False, "non-finite time values"
            if np.ptp(tdec) <= 0.0:
                return False, "collapsed time coordinate"
            if not np.allclose(tdec, reference_tdec, rtol=0.0, atol=1.0e-6):
                return False, "time coordinate differs from source stack"
    except Exception as exc:
        return False, f"could not open existing stack: {exc}"
    return True, "ok"


def build_weighted_input_stack(band_name):
    TEMP_DIR.mkdir(parents=True, exist_ok=True)
    output_path = TEMP_DIR / f"{band_name}_weighted_input.nc"

    error_band = ERROR_BAND_FOR[band_name]
    with stack_for_band(band_name) as data_stack, stack_for_band(error_band) as error_stack:
        ready, reason = _weighted_input_stack_status(output_path, data_stack.tdec)
        if ready and not FORCE_REBUILD_INPUTS:
            return output_path
        if output_path.exists():
            action = "Forced rebuild" if FORCE_REBUILD_INPUTS else f"Rebuilding invalid weighted stack ({reason})"
            print(f"{action}: {output_path}")
            output_path.unlink()

        try:
            _, chunk_ny, chunk_nx = data_stack["chunk_shape"].values
        except KeyError:
            chunk_ny = chunk_nx = 128
        chunks = get_chunks((data_stack.Ny, data_stack.Nx), int(chunk_ny), int(chunk_nx))
        with ice.Stack(str(output_path), mode="w", init_tdec=data_stack.tdec,
                       init_rasterinfo=data_stack.hdr) as output_stack:
            output_stack.init_default_datasets(weights=True, chunks=(1, int(chunk_ny), int(chunk_nx)))
            for islice, jslice in chunks:
                data = data_stack.get_chunk(islice, jslice, key=DATA_KEY).astype(np.float32)
                error = error_stack.get_chunk(islice, jslice, key=DATA_KEY).astype(np.float32)
                weights = uncertainty_to_weights(error)
                invalid = (~np.isfinite(data)) | (~np.isfinite(weights))
                data[invalid] = np.nan
                weights[invalid] = np.nan
                output_stack.set_chunk(islice, jslice, data, key="data")
                output_stack.set_chunk(islice, jslice, weights, key="weights")
    return output_path


def solver_output_dir(band_name):
    return TEMP_DIR / f"{band_name}_solver"

## Run the Full Grid Inversion

This is the cell that launches the joblib-backed inversion. It does nothing until `RUN_FULL_GRID` is set to `True`.

In [ ]:
if not RUN_FULL_GRID:
    print("RUN_FULL_GRID is False. Set it to True after choosing REG_PARAM.")
else:
    for band_name in BANDS_TO_INVERT:
        weighted_input = build_weighted_input_stack(band_name)
        outdir = solver_output_dir(band_name)
        outdir.mkdir(parents=True, exist_ok=True)
        print(f"Running {band_name}: input={weighted_input}, outdir={outdir}")
        with ice.Stack(str(weighted_input)) as stack:
            grid_model = ice.tseries.build_temporal_model(stack.tdec, **MODEL_KWARGS)
            ice.tseries.inversion(
                stack,
                grid_model,
                str(outdir),
                solver_type=SOLVER_TYPE,
                dkey="data",
                nt_out=NT_OUT,
                n_proc=N_PROC,
                regParam=REG_PARAM,
                n_min=N_MIN,
                n_iter=N_ITER,
                n_std=N_STD,
                no_weights=False,
                prior_cov=PRIOR_COV,
            )


## Combine Band Outputs into One NetCDF

The solver writes one component stack per band. This cell combines those solver outputs into a single file with dimensions `(time, band, y, x)`.

In [ ]:
def datetime64_to_epoch_seconds(times):
    epoch = np.datetime64("1970-01-01T00:00:00", "ns")
    return (np.asarray(times).astype("datetime64[ns]") - epoch) / np.timedelta64(1, "s")


def write_combined_output():
    if OUTPUT_NC.exists():
        if not OVERWRITE_OUTPUT:
            raise FileExistsError(f"Set OVERWRITE_OUTPUT=True to replace {OUTPUT_NC}")
        OUTPUT_NC.unlink()

    first_component = solver_output_dir(BANDS_TO_INVERT[0]) / "interp_output_full.h5"
    with ice.Stack(str(first_component)) as first_stack:
        times = first_stack.ds["time"].values
        x = first_stack.ds["x"].values
        y = first_stack.ds["y"].values
        nt, ny, nx = first_stack["data"].shape

    chunks = (1, 1, min(128, ny), min(128, nx))
    with h5netcdf.File(OUTPUT_NC, "w") as fid:
        fid.dimensions["time"] = nt
        fid.dimensions["band"] = len(BANDS_TO_INVERT)
        fid.dimensions["y"] = ny
        fid.dimensions["x"] = nx

        time_var = fid.create_variable("time", ("time",), dtype="f8",
                                       data=datetime64_to_epoch_seconds(times))
        time_var.attrs["units"] = TIME_UNITS
        time_var.attrs["calendar"] = TIME_CALENDAR
        band_dtype = h5py.string_dtype(encoding="utf-8")
        fid.create_variable("band", ("band",), dtype=band_dtype,
                            data=np.asarray(BANDS_TO_INVERT, dtype=object))
        fid.create_variable("y", ("y",), dtype="f8", data=y)
        fid.create_variable("x", ("x",), dtype="f8", data=x)

        variables = {
            key: fid.create_variable(key, ("time", "band", "y", "x"), dtype="f4",
                                     fillvalue=np.float32(np.nan), chunks=chunks)
            for key in COMPONENT_KEYS
        }
        fid.attrs["format"] = "xarray"
        fid.attrs["source_file"] = str(INPUT_NC)
        fid.attrs["regularization_parameter"] = float(REG_PARAM)
        fid.attrs["solver_type"] = SOLVER_TYPE

        for band_index_out, band_name in enumerate(BANDS_TO_INVERT):
            outdir = solver_output_dir(band_name)
            for component in COMPONENT_KEYS:
                component_path = outdir / f"interp_output_{component}.h5"
                with ice.Stack(str(component_path)) as component_stack:
                    variables[component][:, band_index_out, :, :] = component_stack["data"].values.astype(np.float32)

    return OUTPUT_NC


if not RUN_FULL_GRID:
    print("Skipping output combination because RUN_FULL_GRID is False.")
else:
    output_path = write_combined_output()
    print(f"Wrote {output_path}")
    if CLEAN_TEMP:
        shutil.rmtree(TEMP_DIR)
        print(f"Removed {TEMP_DIR}")


## Read the Combined Output with `ice.Stack`

In [ ]:
if OUTPUT_NC.exists():
    with ice.Stack(str(OUTPUT_NC), indexers={"band": 0}) as output_stack:
        full_vx = output_stack["full"]
        print(full_vx)
        frame = output_stack.slice(0, key="full")
        plt.figure(figsize=(10, 6))
        plt.imshow(frame, cmap="viridis")
        plt.colorbar(label="modeled vx")
        plt.title("First modeled vx output frame")
        plt.show()
else:
    print(f"Output does not exist yet: {OUTPUT_NC}")
